# Generate SEND_AR_IND Update SQL

Produces Oracle SQL to set `SEND_AR_IND = 'N'` on the `CORPORATION` table for a provided list of `CORP_NUM` values.
Automatically chunks the list to stay under Oracle's **1000-item IN-clause limit**.

The output SQL contains:
1. A **SELECT** (verification) query — run this first to confirm what will change.
2. An **UPDATE** query — uncomment to execute the actual update.

---

## 1 — Paste your corp nums below

Paste them as a single comma-delimited string of **single-quoted** values, e.g.:

```python
raw_input = "'1111111','2222222','3333333'"
```

In [ ]:
raw_input = (
    "'1111111', '2222222', '3333333'"
)

## 2 — Parse & chunk

In [ ]:
import re
from typing import List

CHUNK_SIZE = 1000  # Oracle IN-clause limit

# Extract all single-quoted values from the raw input
corp_nums: List[str] = re.findall(r"'[^']+'", raw_input)

# Deduplicate while preserving order
seen = set()
unique_corp_nums: List[str] = []
for cn in corp_nums:
    if cn not in seen:
        seen.add(cn)
        unique_corp_nums.append(cn)

# Chunk into groups of CHUNK_SIZE
chunks: List[List[str]] = [
    unique_corp_nums[i:i + CHUNK_SIZE]
    for i in range(0, len(unique_corp_nums), CHUNK_SIZE)
]

print(f'Total values supplied : {len(corp_nums)}')
print(f'Unique values         : {len(unique_corp_nums)}')
print(f'Chunks (≤{CHUNK_SIZE} each)  : {len(chunks)}')

## 3 — Generate SQL

In [ ]:
VALS_PER_LINE = 1000  # how many values per line inside IN()

def build_in_clauses(chunks: List[List[str]], column: str = 'c.CORP_NUM') -> str:
    """Build an OR-chained set of IN clauses from chunks, packed horizontally."""
    parts = []
    for chunk in chunks:
        lines = []
        for i in range(0, len(chunk), VALS_PER_LINE):
            lines.append(','.join(chunk[i:i + VALS_PER_LINE]))
        values = ',\n        '.join(lines)
        parts.append(f'{column} IN (\n        {values}\n    )')
    return '\n    OR '.join(parts)


in_clauses = build_in_clauses(chunks)

sql = f"""-- =============================================================
-- SEND_AR_IND Update Script
-- Generated for {len(unique_corp_nums)} corp(s) in {len(chunks)} chunk(s)
-- =============================================================

-- STEP 1: VERIFY — Run this SELECT first to confirm the rows.
SELECT c.CORP_NUM,
       c.CORP_TYP_CD,
       c.SEND_AR_IND
  FROM CORPORATION c
 WHERE (
    {in_clauses}
)
   AND c.SEND_AR_IND <> 'N'
 ORDER BY c.CORP_NUM;


-- STEP 2: UPDATE — Uncomment the block below once you've verified.
/*
UPDATE CORPORATION c
   SET c.SEND_AR_IND = 'N'
 WHERE (
    {in_clauses}
)
   AND c.SEND_AR_IND <> 'N';

-- Check affected row count
-- SELECT SQL%ROWCOUNT AS rows_updated FROM DUAL;

COMMIT;
*/
"""

print(sql)

## 4 — Save to file (optional)

In [ ]:
output_path = 'generated/send_ar_ind_update.sql'

with open(output_path, 'w') as f:
    f.write(sql)

print(f'SQL written to {output_path}')